# merge_candidates ordering bug — full-dataset eval, not a single example

`rag/tools.py`'s `merge_candidates([exact, fuzzy, semantic])` is plain list
concatenation with de-duplication: every exact result outranks every fuzzy
result, which outranks every semantic result, regardless of how confident any
individual match actually is. A single hand-picked example (`"demode"` →
fuzzy's `decode`/`demote` coincidences beating semantic's on-theme
`Degeneracy`/`Degenerative`) shows the *mechanism*, but proves nothing about
whether fixing it actually helps real queries — that needs the full labeled
eval dataset, not one anecdote.

This notebook runs the real `eval-retrieval` pipeline (real DB, real
embedding model) over the full 96-query labeled set (`data/eval/queries_en
.jsonl` + `queries_ml.jsonl`), comparing the current priority-merge against
Reciprocal Rank Fusion (RRF) — a standard, LLM-free technique for combining
rankings from tools whose raw scores aren't on a comparable scale.

In [1]:
import os

from omegaconf import OmegaConf

import linguaalayam.eval.runner as runner_module
from linguaalayam.database import build_engine, build_session_factory
from linguaalayam.embeddings import EmbeddingService
from linguaalayam.env import load_env
from linguaalayam.eval.dataset import load_dataset
from linguaalayam.eval.metrics import hit_rate, hit_rate_at_1, mrr, tool_breakdown
from linguaalayam.eval.runner import run_eval
from linguaalayam.rag.tools import DictionaryTools, merge_candidates

load_env()

db_cfg = OmegaConf.create({
    "user": os.environ.get("DB_USER", "postgres"),
    "password": os.environ.get("DB_PASSWORD", ""),
    "host": "localhost",
    "port": int(os.environ.get("DB_PORT", "5432")),
    "name": os.environ.get("DB_NAME", "linguaalayam"),
    "pool_size": 2,
    "max_overflow": 2,
})
embed_cfg = OmegaConf.create({
    "model": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "batch_size": 32,
    "normalize": True,
    "quantise": False,
})
engine = build_engine(db_cfg)
sf = build_session_factory(engine)
service = EmbeddingService(embed_cfg)
tools = DictionaryTools(sf, service)

/home/sachin_nandakumar/projects/LinguAalayam/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. The RRF merge function

In [2]:
def merge_candidates_rrf(lists: list[list[dict]], k: int = 60) -> list[dict]:
    """Reciprocal Rank Fusion merge — rank-based, not raw-score-based.

    score(candidate) = sum over every list it appears in of 1 / (k + rank_in_that_list)

    Unlike the current merge_candidates (strict tool priority: exact > fuzzy >
    semantic regardless of confidence), this lets each tool\'s own internal
    ranking compete fairly — a candidate that\'s #1 in semantic\'s own list is
    treated the same as a candidate that\'s #1 in fuzzy\'s own list, instead of
    semantic always losing by default.
    """
    scores: dict[tuple[str, str], float] = {}
    items: dict[tuple[str, str], dict] = {}
    for results in lists:
        for rank, item in enumerate(results, start=1):
            key = (item["source"], item["headword"])
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
            items.setdefault(key, item)
    ranked_keys = sorted(scores, key=lambda key: scores[key], reverse=True)
    return [items[key] for key in ranked_keys]

## 2. The single-example demo first (for intuition only — not evidence)

Reproduces the `"demode"` case discussed in chat: shows the mechanism works
as described. This cell is illustrative, not a basis for any conclusion —
section 3 is the real evidence.

In [3]:
query = "demode"
exact = tools.exact_lookup(query)
fuzzy = tools.fuzzy_lookup(query, top_k=5)
semantic = tools.semantic_lookup(query, top_k=5)

print("fuzzy:   ", [(r["headword"], round(r["score"], 3)) for r in fuzzy])
print("semantic:", [(r["headword"], round(r["score"], 3)) for r in semantic])

baseline_merged = merge_candidates([exact, fuzzy, semantic])
rrf_merged = merge_candidates_rrf([exact, fuzzy, semantic])

print("\ncurrent merge (priority):", [r["headword"] for r in baseline_merged[:5]])
print("RRF merge:               ", [r["headword"] for r in rrf_merged[:5]])

fuzzy:    [('demote', 0.833), ('decode', 0.833)]
semantic: [('Degeneracy', 0.717), ('Degenerative', 0.706), ('Degeneration', 0.706), ('Depolarize', 0.698), ('Demerge', 0.688)]

current merge (priority): ['demote', 'decode', 'Degeneracy', 'Degenerative', 'Degeneration']
RRF merge:                ['demote', 'Degeneracy', 'decode', 'Degenerative', 'Degeneration']


## 3. The real evidence — full labeled dataset, both merge strategies

Runs the actual `eval-retrieval` pipeline (same code path as
`poetry run eval-retrieval`) over all 96 real labeled queries, computing
Hit@1/Hit@k/MRR for both merge strategies. This is what the conclusion is
actually based on, not section 2's single example.

In [4]:
eval_cfg = OmegaConf.create({"top_k": 5, "fuzzy_threshold": 0.75, "fuzzy_limit": 10, "dataset": None})

results_by_label = {}
for label, path in [("EN-input", "../data/eval/queries_en.jsonl"), ("ML-input", "../data/eval/queries_ml.jsonl")]:
    queries = load_dataset(path)

    runner_module.merge_candidates = merge_candidates
    baseline = run_eval(tools, eval_cfg, queries=queries)

    runner_module.merge_candidates = merge_candidates_rrf
    rrf = run_eval(tools, eval_cfg, queries=queries)

    results_by_label[label] = {"queries": queries, "baseline": baseline, "rrf": rrf}

runner_module.merge_candidates = merge_candidates  # restore for anything else in this kernel

In [5]:
for label, data in results_by_label.items():
    baseline, rrf = data["baseline"], data["rrf"]
    n = len(data["queries"])
    print(f"=== {label} ({n} real labeled queries) ===")
    print(
        f"  BASELINE (priority merge): Hit@1={hit_rate_at_1(baseline):.3f}  "
        f"Hit@5={hit_rate(baseline):.3f}  MRR={mrr(baseline):.3f}"
    )
    print(f"    tool attribution: {tool_breakdown(baseline)}")
    print(
        f"  RRF merge:                 Hit@1={hit_rate_at_1(rrf):.3f}  "
        f"Hit@5={hit_rate(rrf):.3f}  MRR={mrr(rrf):.3f}"
    )
    print(f"    tool attribution: {tool_breakdown(rrf)}")
    print()

=== EN-input (50 real labeled queries) ===
  BASELINE (priority merge): Hit@1=0.200  Hit@5=0.200  MRR=0.200
    tool attribution: {'exact': 10, 'fuzzy': 0, 'semantic': 0, 'miss': 40}
  RRF merge:                 Hit@1=0.200  Hit@5=0.200  MRR=0.200
    tool attribution: {'exact': 10, 'fuzzy': 0, 'semantic': 0, 'miss': 40}

=== ML-input (46 real labeled queries) ===
  BASELINE (priority merge): Hit@1=0.217  Hit@5=0.261  MRR=0.236
    tool attribution: {'exact': 10, 'fuzzy': 0, 'semantic': 2, 'miss': 34}
  RRF merge:                 Hit@1=0.217  Hit@5=0.261  MRR=0.236
    tool attribution: {'exact': 10, 'fuzzy': 0, 'semantic': 2, 'miss': 34}



## 4. Which specific queries changed outcome

In [6]:
for label, data in results_by_label.items():
    baseline, rrf = data["baseline"], data["rrf"]
    baseline_hit1 = {b.query: b.hit_at_1 for b in baseline}
    rrf_hit1 = {r.query: r.hit_at_1 for r in rrf}
    improved = [q for q in baseline_hit1 if not baseline_hit1[q] and rrf_hit1[q]]
    regressed = [q for q in baseline_hit1 if baseline_hit1[q] and not rrf_hit1[q]]
    print(f"{label}:")
    print(f"  Hit@1 improved by RRF ({len(improved)}): {improved}")
    print(f"  Hit@1 regressed by RRF ({len(regressed)}): {regressed}")

EN-input:
  Hit@1 improved by RRF (0): []
  Hit@1 regressed by RRF (0): []
ML-input:
  Hit@1 improved by RRF (0): []
  Hit@1 regressed by RRF (0): []


## Conclusion — negative result, and an important one

Across the full 96-query labeled dataset (50 EN, 46 ML), RRF produces **byte-identical
Hit@1/Hit@5/MRR and tool attribution** to the current priority merge, in both languages.
Zero queries changed outcome in either direction.

**Why**: `tool_breakdown` shows the correct headword was never attributed to the `fuzzy`
tool in this dataset at all (only `exact` and `semantic` ever contributed a correct hit).
RRF only changes anything when a weak fuzzy coincidence is burying a correct semantic
match inside the top-k — that situation doesn't occur here. The section-2 `"demode"`
example is real (RRF genuinely reorders those specific candidates), but it isn't
representative of this dataset's actual failure pattern.

**The dominant failure mode is miss, not rank**: 40/50 EN queries and 34/46 ML queries
are missed by every tool, not found-but-ranked-wrong by one. This independently confirms
`research/embedding-quality-eval`'s own prior conclusion ("current miss rates (67-87%)
suggest retrieval recall, not rank order, is the bottleneck") — reached completely
separately, via the reranking investigation rather than this one.

**Practical implication**: `merge_candidates`'s ordering bug is still real and still a
latent risk (a bigger/messier dataset or a different query mix could expose it), but
it is demonstrably **not** the thing costing retrieval quality on real queries today.
Don't prioritize fixing it on the strength of the single illustrative example — the
actual lever to pull is retrieval recall (why do 65-87% of queries find nothing in any
tool's results at all), which is a different, likely bigger investigation.